In [ ]:
import os
import glob
import re
import pandas as pd
import numpy as np
import aqi
from sqlalchemy import create_engine
from sqlalchemy.dialects.postgresql import insert
from dotenv import load_dotenv

In [ ]:
DATA_DIR = "data/air"
load_dotenv()
DB_URL = os.getenv("DB_URL")

NAME_NORMALIZATION = {
    "Bolbes": "Volvi",
    "Ampelokepon-Menemenes": "Ampelokipoi-Menemeni",
    "Ampelokipon": "Ampelokipoi-Menemeni",
    "Thermaikou": "Thermaikos",
    "Khalkedonos": "Chalkidona",
    "Chalkidonos": "Chalkidona",
    "Kordeliou": "Kordelio-Evosmos",
    "Kordeliou-Euosmou": "Kordelio-Evosmos",
    "Lagkada": "Lagkadas",
    "Neapoles-Sykeon": "Neapoli-Sykies",
    "Neapoles-Sukeon": "Neapoli-Sykies",
    "Neapoli": "Neapoli-Sykies",
    "Pavlou_Mela": "Pavlos Melas",
    "Paulou_Mela": "Pavlos Melas",
    "Thermes": "Thermi",
    "Thessalonikes": "Thessaloniki",
    "Pulaia": "Pylaia-Chortiatis",
    "Pulaias-Khortiate": "Pylaia-Chortiatis",
    "Oraiokastrou": "Oraiokastro"
}

In [ ]:
def calculate_daily_aqi(daily_df):
    sub_indices = []

    try:
        #O3- peak 8-hour rolling average
        if 'o3' in daily_df.columns:
            o3_8h_peak = daily_df['o3'].rolling(8, min_periods=6).mean().max()
            if pd.notna(o3_8h_peak):
                o3_ppm = (o3_8h_peak * 24.45) / (48.00 * 1000)  #µg/m³ → ppm
                sub_indices.append(aqi.to_iaqi(aqi.POLLUTANT_O3_8H, str(o3_ppm)))

        #CO- peak 8-hour rolling average
        if 'co' in daily_df.columns:
            co_8h_peak = daily_df['co'].rolling(8, min_periods=6).mean().max()
            if pd.notna(co_8h_peak):
                co_ppm = (co_8h_peak * 24.45) / (28.01 * 1000)  #µg/m³ → ppm
                sub_indices.append(aqi.to_iaqi(aqi.POLLUTANT_CO_8H, str(co_ppm)))

        #NO2- 1 hour peak
        if 'no2' in daily_df.columns:
            no2_peak = daily_df['no2'].max()
            if pd.notna(no2_peak):
                no2_ppb = (no2_peak * 24.45) / 46.01  #µg/m³ → ppb
                sub_indices.append(aqi.to_iaqi(aqi.POLLUTANT_NO2_1H, str(no2_ppb)))

        #SO2, 1 hour peak
        if 'so2' in daily_df.columns:
            so2_peak = daily_df['so2'].max()
            if pd.notna(so2_peak):
                so2_ppb = (so2_peak * 24.45) / 64.06  #µg/m³ → ppb
                sub_indices.append(aqi.to_iaqi(aqi.POLLUTANT_SO2_1H, str(so2_ppb)))

        return float(max(sub_indices)) if sub_indices else np.nan

    except Exception:
        return np.nan

In [ ]:
file_pattern = os.path.join(DATA_DIR, "*", "municipality_of_*_pollutants_conc_timeseries-yearly_*.csv")
all_files = glob.glob(file_pattern)

if not all_files:
    print(f"No CSV files found matching the pattern in '{DATA_DIR}' subfolders!")

all_monthly_summaries = []

for file_path in all_files:
    filename = os.path.basename(file_path)

    match = re.search(r"municipality_of_(.+)_pollutants_conc_timeseries-yearly", filename)
    if not match:
        continue

    raw_municipality = match.group(1).title()
    municipality = NAME_NORMALIZATION.get(raw_municipality, raw_municipality)

    df = pd.read_csv(file_path)

    # Normalize column names: drop the _conc suffix from newer datasets
    df.rename(columns={
        'co_conc':  'co',
        'no2_conc': 'no2',
        'so2_conc': 'so2',
        'o3_conc':  'o3'
    }, inplace=True)

    df['date'] = pd.to_datetime(df['time'])
    df = df.sort_values('date').set_index('date')

    #Groups by calendar date,applies EPA averaging windows inside each group
    daily_aqi_series = (
        df.groupby(df.index.date)
          .apply(calculate_daily_aqi)
    )
    daily_aqi_df = daily_aqi_series.reset_index()
    daily_aqi_df.columns = ['day', 'daily_aqi']
    daily_aqi_df['day'] = pd.to_datetime(daily_aqi_df['day'])

    #Aggregates daily AQI to monthly mean
    monthly_summary = (
        daily_aqi_df
        .groupby([
            daily_aqi_df['day'].dt.year.rename('year'),
            daily_aqi_df['day'].dt.month.rename('month')
        ])['daily_aqi']
        .mean()
        .reset_index()
    )
    monthly_summary.rename(columns={'daily_aqi': 'mean_aqi'}, inplace=True)
    monthly_summary['mean_aqi'] = monthly_summary['mean_aqi'].round(0)
    monthly_summary['municipality'] = municipality

    final_df = monthly_summary[['municipality', 'year', 'month', 'mean_aqi']]
    final_df = final_df.dropna(subset=['mean_aqi'])

    all_monthly_summaries.append(final_df)

In [ ]:
if all_monthly_summaries:
    master_df = pd.concat(all_monthly_summaries, ignore_index=True)
    print(f"Total rows: {len(master_df)}")
    print(master_df.head())
else:
    print("No data to process.")

In [ ]:
def insert_do_nothing(table, conn, keys, data_iter):
    #Zips column names and data rows into a list of dictionaries
    data = [dict(zip(keys, row)) for row in data_iter]

    insert_stmt = insert(table.table).values(data)
    do_nothing_stmt = insert_stmt.on_conflict_do_nothing(
        index_elements=['municipality', 'year', 'month']
    )

    result = conn.execute(do_nothing_stmt)
    return result.rowcount


sync_db_url = DB_URL.replace("+asyncpg", "")
engine = create_engine(sync_db_url)

master_df.to_sql(
    'historical_aqi',
    engine,
    if_exists='append',
    index=False,
    method=insert_do_nothing
)